### Aula: Random Forest

### Introdução
- Desenvolvido por Tin Kam Ho em 1995, o Random Forest (Floresta Aleatória) é uma técnica poderosa de ML
  
- Cria múltiplas árvores de decisão durante o treinamento e as combina por meio de votação para obter predições mais robustas

- O algoritmo usa a amostragem de exemplos com reposição do bagging (ou, às vezes, pasting) combinada com a seleção aleatória de atributos

- Reduz a correlação entre as árvores, tornando o modelo menos propenso ao overfitting
      

### Como funciona a Random Forest
- A construção de cada árvore ocorre da seguinte maneira:
  1. Seleção Aleatória de Dados: Cada árvore é treinada com aproximadamente 2/3 (63.2%) dos dados de treinamento totais, selecionados aleatoriamente com reposição do conjunto original. Essa amostra constitui o conjunto de treinamento para a árvore em questão.
  2. Seleção Aleatória de Características: Um subconjunto de características preditoras é escolhido aleatoriamente dentre todas as características preditoras disponíveis. A melhor divisão utilizando esse subconjunto de características é então utilizada para dividir o nó atual da árvore.
  3. Cálculo do Erro out of bag (OOB): Para cada árvore na floresta, os dados não utilizados durante o treinamento (dados out of bag, 36.8%) são utilizados para calcular uma medida de erro conhecida como erro out of bag (OOB). O erro OOB é agregado de todas as árvores para determinar a taxa de erro geral da floresta.
  4. Cada árvore fornece uma classificação e dizemos que a árvore "vota" para essa classe. A floresta escolhe a classificação que recebe mais votos.








### Implementação usando sklearn
- Em vez de criar um ``BagginClassifier`` e passá-lo em um ``DecisionTreeClassifier``, pode-se usar a classe ``RandomForestClassifier``, que é otimizada para Árvores de Decisão
- Para problemas de Regressão também tem a classe ``RandomForestRegressor``

<b>1.Importando as bibliotecas</b>

In [ ]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

<b>2. Carregando o conjunto de dados ``make_moons`` </b>
- Principais parâmetros
    - n_samples: O número total de amostras a serem geradas. Este parâmetro determina o tamanho do conjunto de dados.
    - noise: O desvio padrão do ruído gaussiano adicionado aos dados. Quanto maior o valor do ruído, mais difícil é a tarefa de classificação, pois os dados se tornam mais sobrepostos
    - random_state: O estado do gerador de números aleatórios para garantir a reprodutibilidade dos resultados

In [ ]:
# Carregando o conjunto de dados
X, y = make_moons(n_samples=1000, noise=0.30, random_state=42)

<b>3. Visualização dos dados Gerados </b>

In [ ]:
# Separar as classes
class_0 = X[y == 0]
class_1 = X[y == 1]

# Plotar os dados
plt.figure(figsize=(8, 6))
plt.scatter(class_0[:, 0], class_0[:, 1], label='Classe 0', c='blue', marker='o')
plt.scatter(class_1[:, 0], class_1[:, 1], label='Classe 1', c='red', marker='s')
plt.title('Conjunto de Dados make_moons')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True)
plt.show()

<b>4. Divisão estratificada em treino e teste do conjunto de dados</b>
- O parâmetro ``test_size=0.2`` define a proporção dos dados que serão utilizados como conjunto de teste (neste caso, 20%)
- o parâmetro ``stratify=y`` garante que a divisão seja estratificada, ou seja, as proporções de classes em $y$ sejam mantidas nos conjuntos de treinamento e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

<b>5. Criando e treinando o classificador RandomForest</b>

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=500,
                                 max_leaf_nodes=16,
                                 n_jobs=-1)
rf_clf.fit(X_train, y_train)


<b>6. Avaliando o desempenho do classificador RandomForest</b>

In [ ]:
tree_clf = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_clf.fit(X_train, y_train)

accuracy_dt = tree_clf.score(X_test, y_test)
print(f'Acurácia da Árvore de Decisão: {accuracy_dt}')

accuracy_rf = rf_clf.score(X_test, y_test)
print(f'Acurácia do classificador Random Forest: {accuracy_rf}')

- Salvo raras exceções, ``RandomForestClassifier`` tem a maioria dos hiperparâmetros de um ``DecisionTreeClassifier`` e todos os hiperparâmetros de um ``BaggingClassifier``
- O algoritmo Random Forest introduz mais aleatoriedade ao cultivar árvores, procurando pela melhor característica em um subconjunto aleatório de características
- Isso aumenta a diversidade das árvores, reduzindo a variância do modelo e geralmente resultando em um modelo melhor

### Importância das características
- Ao analisar uma única Árvore de Decisão:
    - Características importantes provavelmente aparecerão mais próximas à raiz da árvore.
    - Características menos importantes frequentemente aparecerão mais próximas às folhas ou podem nem aparecer.
- É possível obter uma estimativa da importância de uma característica:
    - Calculando a profundidade média na qual ela aparece em todas as árvores da floresta.
- O Scikit-Learn calcula automaticamente a importância de cada característica após o treinamento:
    - Acesso ao resultado usando a variável ``feature_importances_``.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
rf_clf = RandomForestClassifier(n_estimators=500, random_state=42)
rf_clf.fit(iris.data, iris.target)

for score, name in zip(rf_clf.feature_importances_, iris.data.columns):
    print(round(score, 2), name)

- Exemplo com o ``RandomForestClassifier`` no conjunto de dados iris:
    - As características mais importantes são o comprimento (44%) e a largura (42%) da pétala.
    - O comprimento e a largura do sépala são relativamente menos importantes em comparação (11% e 2%, respectivamente).

### Resumo e Considerações Finais
- Exploramos a técnica de Random Forest, uma abordagem poderosa de aprendizado de máquina para classificação e regressão
- Iniciamos entendendo o conceito por trás do Random Forest, que é um ensemble de árvores de decisão
- Exploramos como o algoritmo funciona, destacando a aleatoriedade introduzida durante o treinamento das árvores e a votação para classificação
- Demonstramos a implementação prática do Random Forest usando a biblioteca Scikit-Learn no Python, incluindo treinamento, previsões e avaliação do modelo
- Discutimos a importância das características e como o ``RandomForestClassifier`` pode calcular automaticamente a importância de cada característica após o treinamento
- Vantagens:
    - Capacidade de aprender limites de decisão não lineares:
        - Random Forest é um algoritmo de aprendizagem conjunto que usa múltiplas árvores de decisão para fazer previsões.
        - Pode modelar relacionamentos complexos e não lineares entre atributos e o atributo alvo.
    - Alta precisão:
        - Reduz o problema de overfitting nas árvores de decisão e ajuda a melhorar a precisão.
        - Reduz a variação da previsão em comparação com a árvore de decisão única.
    - Flexível e robusto:
        - Random Forest pode lidar com uma ampla variedade de tipos de dados, incluindo dados numéricos e categóricos.
        - Pode lidar com valores discrepantes e valores ausentes e não requer escalonamento de características, pois usa abordagem baseada em regras em vez de cálculo de distância.
    - Importância das características:
        - Random Forest fornece informações sobre a importância de cada característica nos dados, o que pode ser muito útil na compreensão dos padrões subjacentes.
    - Processamento paralelo:
        - Árvores podem ser criadas em paralelo, pois não há dependência entre iterações, o que agiliza o tempo de treinamento.
            
- Desvantagens:
    - Interpretabilidade:
        - Menos interpretável do que uma única árvore de decisão, uma vez que a previsão não pode ser explicada por apenas um diagrama.
        - No entanto, a importância da variável ainda pode ser extraída.
    - Complexidade computacional:
        - Random Forest pode ser computacionalmente cara, especialmente quando se trabalha com grandes conjuntos de dados
        - Requer muita memória, o que pode ser uma restrição ao trabalhar com recursos limitados.
    - Sensibilidade ao ruído:
        - Embora Random Forest seja resistente ao overfitting, isso ainda pode ocorrer em certos casos, principalmente ao trabalhar com dados ruidosos

### <p style="color:blue">Próxima aula: Boosting</p>